# ⚾ 재대결 회피의 "선택가치": RVAE의 구조적 상쇄를 피한 재검증

* **작성자:** DY
* **작성일:** 2026-10-05
* **배경:** 노트북 07의 RVAE 기반 모델은 결과가 비유의했습니다. 그런데 RVAE = 실제 결과 − 예측(실제로
  **선택한 구종** 기준)이라서, GBM의 `MODEL_FEATURES`에 `pitch_type`이 그대로 들어가 있으면 "그 구종을
  고른 것 자체의 이득"이 예측값 쪽으로 흡수되고, 빼고 나면 사라집니다 — GBM 설명력을 아무리 올려도 고쳐지지
  않는 구조적 상쇄입니다. 게다가 (D)/(E) 모델의 통제변수(결정구 시점의 balls/strikes/pitch_family)는
  재대결 타석이 어떻게 진행됐는지의 **결과물**이라, "회피"라는 타석 전체 수준 선택의 효과를 통제하면서
  같이 지워버릴 위험이 있습니다(post-treatment bias).
* **이 노트북이 다르게 재는 것 — 선택가치 (counterfactual swap).** "그 구종을 고른 것 자체가 좋은
  선택이었는가"를 직접 검정합니다. 노트북 03의 GBM을 그대로 쓰되, 같은 상황에서 `pitch_type`만 "실제로
  던진 구종" → "맞은 구종(재사용 가정)"으로 바꿔 두 번 예측합니다.
  `선택가치 = 예측(재사용 가정) − 예측(실제 선택)`. 양수면 실제 선택이 재사용보다 나았다는 뜻이고,
  실제 결과(woba_value)를 전혀 쓰지 않는 순수 모델-내적 비교라 RVAE의 상쇄 문제가 없습니다.
  공변량에서 `balls`/`strikes`(결정구 시점, post-treatment)와 `pitch_family`(선택가치의 정의 자체와
  순환되므로)를 **의도적으로 제외**합니다 — `outs_when_up`/`score_diff`/`platoon_match`/`risp`/
  `runners_n`/`season`만 통제합니다(이들은 타석이 어떻게 진행됐든 거의 바뀌지 않는, 타석 시작 이전
  상태에 가깝습니다).
* **설계:** 이 노트북은 대화로 합의된(bounded) 작업으로, 별도 spec 문서 없이 노트북 03·07의 기존 인프라를
  그대로 재사용해 진행합니다.

In [1]:
# 1. 경로 설정 (src 폴더 접근용)
import sys, os
sys.path.append(os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

from src.analysis.avoidance_stats import compute_season_usage_rate
from src.analysis.glmer_runner import (
    check_convergence,
    extract_lmer_fixed_effects,
    extract_variance_components,
    fit_lmer,
)
from src.analysis.mixed_effects_model import compute_icc_gaussian
from src.analysis.run_same_batter_rematch_analysis import build_xbh_rematch
from src.preprocessing.pitch_family import map_pitch_family

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

TRAIN_SEASONS = (2021, 2022, 2023, 2024, 2025)
VALID_SEASON = 2026
RANDOM_STATE = 20261003

Error importing in API mode: ImportError("dlopen(/Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib\n  Referenced from: <9F0E20C3-D782-31B5-943E-66F62BCE77E8> /Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


## 1-7. 노트북 07과 같은 인프라 재사용

팀 CSV 로딩 → 상황 피처 → 재대결 이벤트(27,391건) → GBM 학습 테이블 → 전체 구종 포함 GBM 학습 →
재대결 결정구 매칭 → `model_data`(27,245건, `avoided` 포함)까지, 노트북 07과 완전히 같은 과정입니다.

In [2]:
from src.preprocessing.research_sample import KEY_COLUMNS, list_research_csvs
from src.preprocessing.build_next_ab_dataset import EXTRA_BASE_HIT_EVENTS

LOAD_COLUMNS = [
    'game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'pitcher', 'player_name', 'batter',
    'stand', 'p_throws', 'pitch_type', 'events', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b', 'home_score', 'away_score',
    'woba_value', 'woba_denom',
]


def load_team_csv() -> pd.DataFrame:
    frames = [
        pd.read_csv(path, usecols=LOAD_COLUMNS, encoding='utf-8-sig', low_memory=False)
        for path in list_research_csvs()
    ]
    pitches = pd.concat(frames, ignore_index=True)
    pitches['season'] = pd.to_datetime(pitches['game_date']).dt.year
    return pitches


pitches = load_team_csv()
assert len(pitches) == 3_592_302 and pitches['pitcher'].nunique() == 1_067
assert not pitches.duplicated(KEY_COLUMNS).any()
print(f'{len(pitches):,}행, 투수 {pitches["pitcher"].nunique():,}명')

3,592,302행, 투수 1,067명


In [3]:
SITUATION_COLUMNS = ['runners_n', 'risp', 'score_diff', 'platoon_match']


def add_situation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    on1, on2, on3 = out['on_1b'].notna(), out['on_2b'].notna(), out['on_3b'].notna()
    out['runners_n'] = on1.astype(int) + on2.astype(int) + on3.astype(int)
    out['risp'] = (on2 | on3).astype(int)
    out['score_diff'] = np.where(
        out['inning_topbot'] == 'Top', out['home_score'] - out['away_score'], out['away_score'] - out['home_score']
    )
    out['platoon_match'] = np.where(
        out['stand'].isna() | out['p_throws'].isna(), np.nan, (out['stand'] == out['p_throws']).astype(float)
    )
    return out


_t = pd.DataFrame({
    'on_1b': [np.nan, 1.0, np.nan], 'on_2b': [np.nan, np.nan, 2.0], 'on_3b': [np.nan, np.nan, np.nan],
    'inning_topbot': ['Top', 'Bot', 'Top'], 'home_score': [3, 5, 2], 'away_score': [3, 2, 4],
    'stand': ['L', 'R', 'R'], 'p_throws': ['R', 'R', 'L'],
})
_r = add_situation_columns(_t)
assert _r['runners_n'].tolist() == [0, 1, 1]
assert _r['risp'].tolist() == [0, 0, 1]
assert _r['score_diff'].tolist() == [0, -3, -2]
assert _r['platoon_match'].tolist() == [0.0, 1.0, 0.0]
print('add_situation_columns 테스트 통과')

pitches = add_situation_columns(pitches)

add_situation_columns 테스트 통과


In [4]:
season_usage = compute_season_usage_rate(pitches)
xbh = build_xbh_rematch(pitches, season_usage)
xbh = xbh[xbh['has_next_ab']].copy()
assert len(xbh) == 27_391, len(xbh)
print(f'재대결 이벤트 {len(xbh):,}건')

2026-10-05 20:21:35,682 [INFO] 장타 70319건 중 같은 타자·같은 투수 재대결 27391건 (39.0%)
        xbh_total  same_batter_rematch  rematch_share
season                                               
2021        12709                 4688         0.3689
2022        12186                 4623         0.3794
2023        13137                 5280         0.4019
2024        12427                 4977         0.4005
2025        12394                 4916         0.3966
2026         7466                 2907         0.3894


재대결 이벤트 27,391건


In [5]:
MODEL_FEATURES = [
    'pitch_type', 'stand', 'p_throws', 'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff',
    'season_usage_rate',
]
CATEGORICAL_FEATURES = ['pitch_type', 'stand', 'p_throws']


def build_decisive_pitch_table(pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna() & pitches_sit['pitch_type'].notna()].copy()
    decisive = decisive.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    for col in CATEGORICAL_FEATURES:
        decisive[col] = decisive[col].astype('category')
    return decisive


decisive_table = build_decisive_pitch_table(pitches, season_usage)
assert decisive_table['season_usage_rate'].notna().mean() > 0.99
print(f'결정구 {len(decisive_table):,}건, season_usage_rate 결측 {decisive_table["season_usage_rate"].isna().sum()}건')

결정구 917,717건, season_usage_rate 결측 0건


In [6]:
def fit_run_value_model(table: pd.DataFrame, seasons) -> HistGradientBoostingRegressor:
    train = table[table['season'].isin(seasons)]
    model = HistGradientBoostingRegressor(
        categorical_features='from_dtype', random_state=RANDOM_STATE, max_iter=200, early_stopping=True,
    )
    model.fit(train[MODEL_FEATURES], train['woba_value'])
    return model


train_set = decisive_table[decisive_table['season'].isin(TRAIN_SEASONS)]
valid_set = decisive_table[decisive_table['season'] == VALID_SEASON]
model_cv = fit_run_value_model(decisive_table, TRAIN_SEASONS)
pred_valid = model_cv.predict(valid_set[MODEL_FEATURES])
model_r2 = r2_score(valid_set['woba_value'], pred_valid)
print(f'(전체 피처) GBM 검증 R²: {model_r2:.4f}')
assert model_r2 > 0

final_model = fit_run_value_model(decisive_table, decisive_table['season'].unique())
print('(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀')

(전체 피처) GBM 검증 R²: 0.0568


(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀


In [7]:
def find_rematch_outcome_pitch(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna()][
        ['game_pk', 'pitcher', 'season', 'at_bat_number', *MODEL_FEATURES, 'platoon_match', 'events', 'woba_value']
    ]
    keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'reused_same_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    out = keys.merge(decisive, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')
    return out


REQUIRED_OUTCOME_COLUMNS = sorted(set(['events', 'woba_value', 'platoon_match', *MODEL_FEATURES]))


def build_avoidance_outcome_dataset(found: pd.DataFrame) -> pd.DataFrame:
    out = found.copy()
    out['pitch_family'] = out['pitch_type'].map(map_pitch_family)
    out = out.dropna(subset=[*REQUIRED_OUTCOME_COLUMNS, 'pitch_family', 'reused_same_type'])
    out['avoided'] = (1 - out['reused_same_type']).astype(int)
    out['allowed_xbh_again'] = out['events'].isin(EXTRA_BASE_HIT_EVENTS).astype(int)
    out['pitcher'] = out['pitcher'].astype(str)
    return out


found = find_rematch_outcome_pitch(xbh, decisive_table)
found_with_pa = found.dropna(subset=['woba_value'])
model_data = build_avoidance_outcome_dataset(found_with_pa)
assert len(model_data) == 27_245, len(model_data)
print(f'모델링에 쓸 재대결: {len(model_data):,}건, 투수 {model_data["pitcher"].nunique()}명 (노트북 07과 동일해야 함)')

모델링에 쓸 재대결: 27,245건, 투수 695명 (노트북 07과 동일해야 함)


## 8. 반사실(counterfactual) 구종 붙이기

각 재대결 이벤트에 "맞은 구종"(`hit_pitch_type`, 재사용했다면 던졌을 구종)과, 그 투수·시즌에서 그 구종을
쓴 비율(`hit_pitch_season_usage_rate`)을 붙입니다. 그 시즌에 한 번도 안 던진 구종이면 사용률은 0으로
채웁니다(결측이 아니라 "그 시즌엔 안 씀"이 맞는 값이므로).

In [8]:
def attach_counterfactual_pitch_type(model_data_: pd.DataFrame, xbh_: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    cf_keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'hit_pitch_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    cf_keys['pitcher'] = cf_keys['pitcher'].astype(str)
    out = model_data_.merge(cf_keys, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')

    usage = season_usage_.rename(
        columns={'pitch_type': 'hit_pitch_type', 'season_usage_rate': 'hit_pitch_season_usage_rate'}
    ).copy()
    usage['pitcher'] = usage['pitcher'].astype(str)
    out = out.merge(
        usage[['pitcher', 'season', 'hit_pitch_type', 'hit_pitch_season_usage_rate']],
        on=['pitcher', 'season', 'hit_pitch_type'], how='left',
    )
    out['hit_pitch_season_usage_rate'] = out['hit_pitch_season_usage_rate'].fillna(0.0)
    return out

In [9]:
_md_fix = pd.DataFrame({
    'game_pk': [100, 100], 'pitcher': ['10', '10'], 'at_bat_number': [5, 9], 'season': [2021, 2021],
})
_xbh_fix = pd.DataFrame({
    'game_pk': [100, 100], 'pitcher': [10, 10], 'next_at_bat_number': [5.0, 9.0],
    'hit_pitch_type': ['SL', 'CH'],
})
_su_fix = pd.DataFrame({'pitcher': [10], 'season': [2021], 'pitch_type': ['SL'], 'season_usage_rate': [0.35]})
_cf = attach_counterfactual_pitch_type(_md_fix, _xbh_fix, _su_fix)
assert _cf['hit_pitch_type'].tolist() == ['SL', 'CH']
assert np.isclose(_cf.loc[_cf['at_bat_number'] == 5, 'hit_pitch_season_usage_rate'].iloc[0], 0.35)
assert _cf.loc[_cf['at_bat_number'] == 9, 'hit_pitch_season_usage_rate'].iloc[0] == 0.0
print('attach_counterfactual_pitch_type 테스트 통과')

attach_counterfactual_pitch_type 테스트 통과


In [10]:
model_data = attach_counterfactual_pitch_type(model_data, xbh, season_usage)
assert model_data['hit_pitch_type'].notna().all()
print('반사실 구종 결측 0건 확인')

반사실 구종 결측 0건 확인


## 9. 선택가치: counterfactual swap

`선택가치 = 예측(맞은 구종을 재사용했다고 가정) − 예측(실제로 던진 구종)`. 양수면 실제 선택이 재사용보다
나았다는 뜻. 실제 결과(woba_value)를 쓰지 않는 순수 모델-내적 비교입니다.

In [11]:
def compute_selection_value(df: pd.DataFrame, model) -> pd.Series:
    actual_pred = model.predict(df[MODEL_FEATURES])
    cf = df[MODEL_FEATURES].copy()
    cf['pitch_type'] = df['hit_pitch_type'].astype(df['pitch_type'].dtype)
    cf['season_usage_rate'] = df['hit_pitch_season_usage_rate']
    cf_pred = model.predict(cf)
    return pd.Series(cf_pred - actual_pred, index=df.index)

In [12]:
class _LinearStub:
    def predict(self, X):
        return np.where(X['pitch_type'].astype(str) == 'SL', 0.5, 0.2)


_cat_dtype = pd.CategoricalDtype(categories=['SL', 'FF', 'CH'])
_df_sv = pd.DataFrame({
    'pitch_type': pd.Categorical(['FF', 'SL'], dtype=_cat_dtype), 'hit_pitch_type': ['SL', 'SL'],
    'season_usage_rate': [0.4, 0.5], 'hit_pitch_season_usage_rate': [0.3, 0.5],
    'stand': ['R', 'R'], 'p_throws': ['R', 'R'], 'balls': [0, 0], 'strikes': [0, 0],
    'outs_when_up': [0, 0], 'runners_n': [0, 0], 'risp': [0, 0], 'score_diff': [0, 0],
})
_sv = compute_selection_value(_df_sv, _LinearStub())
assert np.allclose(_sv.to_numpy(), [0.3, 0.0])
print('compute_selection_value 테스트 통과')

compute_selection_value 테스트 통과


In [13]:
model_data['selection_value'] = compute_selection_value(model_data, final_model)
_same_type_mask = model_data['pitch_type'].astype(str) == model_data['hit_pitch_type'].astype(str)
print('검산 -- 결정구가 맞은 구종과 같은 행(재사용)의 선택가치 평균(0이어야 함):',
      model_data.loc[_same_type_mask, 'selection_value'].mean())
display(model_data.groupby('avoided')['selection_value'].describe().round(4))

FORMULA_SELECTION = (
    'selection_value ~ avoided + outs_when_up + score_diff + platoon_match + risp + runners_n '
    '+ factor(season) + (1 | pitcher)'
)
fit_lmer(model_data, FORMULA_SELECTION)
print('=== 수렴 상태 ===')
print(check_convergence())
fixed_selection = extract_lmer_fixed_effects()
print('=== 고정효과 (avoided 포함) ===')
display(fixed_selection.round(4))
vc_selection = extract_variance_components('pitcher')
print('분산 성분:', vc_selection)
if vc_selection['pitcher_intercept'] is None or vc_selection['residual'] is None:
    icc_selection = None
    print('ICC 계산 불가')
else:
    icc_selection = compute_icc_gaussian(vc_selection['pitcher_intercept'], vc_selection['residual'])
    print(f'ICC: {icc_selection:.4f}')

검산 -- 결정구가 맞은 구종과 같은 행(재사용)의 선택가치 평균(0이어야 함): 0.0


,count,mean,std,min,25%,50%,75%,max
avoided,,,,,,,,
0,12440.0,0.0015,0.0182,-0.1904,0.0000,0.0000,0.0070,0.1514
1,14805.0,0.0040,0.0308,-0.3122,-0.0137,0.0041,0.0215,0.3239


2026-10-05 20:22:04,110 [WARNING] R callback write-console: Loading required package: Matrix
  


=== 수렴 상태 ===
OK: no convergence warnings; isSingular=False
=== 고정효과 (avoided 포함) ===


,term,estimate,std_error,t_value,ci_low,ci_high,p_value
0,(Intercept),0.0018,0.0005,3.7513,0.0009,0.0028,0.0002
1,avoided,0.0025,0.0003,7.9832,0.0019,0.0031,0.0000
2,outs_when_up,-0.0001,0.0002,-0.5790,-0.0005,0.0003,0.5626
3,score_diff,0.0001,0.0001,1.2665,-0.0000,0.0002,0.2053
4,platoon_match,-0.0009,0.0003,-2.8141,-0.0015,-0.0003,0.0049
5,risp,-0.0005,0.0006,-0.7940,-0.0016,0.0007,0.4272
6,runners_n,0.0003,0.0003,0.8621,-0.0004,0.0009,0.3886
7,factor(season)2022,-0.0004,0.0005,-0.8160,-0.0015,0.0006,0.4145
8,factor(season)2023,0.0002,0.0005,0.3330,-0.0009,0.0012,0.7391
9,factor(season)2024,-0.0000,0.0005,-0.0090,-0.0011,0.0010,0.9928


분산 성분: {'pitcher_intercept': 1.8517348806698918e-06, 'residual': 0.0006657306123836487}
ICC: 0.0028


## 10. 해석

* **결과.** `avoided` 추정치 **+0.0025** [95% CI 0.0019, 0.0031], **p < 0.0001** (n=27,245, t=7.98). 0을
  한참 벗어난, 방향이 뚜렷한 효과입니다. 검산(결정구가 맞은 구종과 같은 행, 즉 재사용한 경우)의 선택가치
  평균이 정확히 0.0으로 나온 것도 구현이 맞다는 걸 확인해 줍니다.
* **방향과 의미.** `선택가치 = 예측(맞은 구종 재사용 가정) − 예측(실제 선택)`이 양수라는 건, GBM이 학습한
  "이 상황에서 이 구종이면 기대 피해가 얼마"라는 관계를 기준으로, **회피한 선택이 재사용하는 것보다
  체계적으로 더 낮은 기대 피해를 가진 구종이었다**는 뜻입니다. 이건 실제 결과(woba_value)를 전혀 쓰지
  않는 순수 모델-내적 비교라, 노트북 07의 RVAE가 갖고 있던 구조적 상쇄(선택한 구종이 baseline 피처에
  들어가 있어서 선택의 이득이 흡수되는 문제) 자체가 없는 지표입니다.
* **크기.** 다만 효과 크기(+0.0025 woba)는 작습니다. 노트북 07의 RVAE 표준편차(~0.52)와 비교하면 약
  1/200 수준이라, 실제 개별 타석의 결과 변동(실행·운)에 비해 선택 하나가 보태는 몫은 작다는 뜻입니다.
  통계적으로 유의하다는 것과 실용적으로 큰 효과라는 것은 다른 이야기입니다 — 표본이 27,245건으로 커서
  이 정도 작은 효과도 매우 정밀하게(SE=0.0003) 검출된 것입니다.
* **한계.** GBM의 `pitch_type` 효과는 타자별 매치업이 아니라 "이 상황에서 이 구종군이 평균적으로 얼마나
  위험한가"라는 리그 평균 수준입니다(`MODEL_FEATURES`에 타자 식별 정보가 없음). 그래서 이 결과는 "이
  특정 타자를 상대로 영리하게 피했다"는 매치업 차원의 증거가 아니라, "평균적으로 덜 위험한 구종으로
  바꿨다"는 선택 차원의 증거로 읽어야 합니다. 또한 `season_usage_rate`를 반사실(재사용 가정) 구종에도
  그 투수의 실제 시즌 구사율로 채웠기 때문에(그 시즌에 한 번도 안 던진 구종이면 0), 구사율이 아주 낮은
  구종에 대한 GBM의 예측은 외삽(extrapolation)에 가까울 수 있습니다.
* **ICC.** 투수 간 분산 비율은 위 출력(`ICC: ...`)대로 노트북 07의 (D)/(E)와 마찬가지로 매우 작습니다 —
  "그 투수가 누구인지"는 선택가치 자체도 거의 설명하지 못합니다(공변량 통제 후 남는 투수 고유의 차이가
  거의 없다는 뜻).